In [1]:
import datetime
import logging  # ログ制御用
import warnings  # 警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Node.js Theme Palette)
# =====================================================================
def show_market_performances(market_dict, title="Market Performance Overview"):
    today = datetime.date.today()
    # 1年分（約370日）のデータを取得して過去騰落率を正確に計算
    start_date = today - datetime.timedelta(days=370)
    
    # Node.js テーマ風のタイトルヘッダー (Dark Background & Neon Green Text)
    title_html = f"""
    <div style='
        background-color: #181818; 
        border-left: 5px solid #68a063; 
        padding: 12px 18px; 
        border-radius: 4px; 
        margin-bottom: 25px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.2);
    '>
        <h2 style='
            margin: 0; 
            color: #539e43; 
            font-family: "Consolas", "Courier New", Monaco, monospace; 
            font-size: 22px;
            letter-spacing: 0.5px;
        '>⬢ {title}</h2>
        <span style='
            font-size: 12px; 
            color: #888888; 
            font-family: monospace;
        '>System Status: ONLINE | As of {today.strftime('%Y-%m-%d')}</span>
    </div>
    """
    display(HTML(title_html))
    
    # 数値フォーマット設定
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # Node.jsテーマに合わせたハイライト関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        # プラス（上昇）：Node.jsのネオングリーン背景
        if val >= 10:
            return 'background-color: #1e3a1e; color: #41d888; font-weight: bold; border-radius: 2px;'
        # マイナス（下落）：ダークレッド/コーラル系
        elif val <= -10:
            return 'background-color: #3b1818; color: #ff6b6b; font-weight: bold; border-radius: 2px;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # セクションヘッダー（Node Dark Gray + Green Accent）
        section_html = f"""
        <div style='
            margin-top: 20px; 
            margin-bottom: 8px; 
            color: #215732; 
            font-family: "Consolas", monospace; 
            font-weight: bold;
            font-size: 15px;
            border-bottom: 1px solid #333333;
            padding-bottom: 4px;
        '>
            <span style='color: #68a063;'>❯</span> {market_name}
        </div>
        """
        display(HTML(section_html))
        
        for name, ticker in ticker_dict.items():
            try:
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # テーブル全体のスタイル定義（Dark Node UI Style）
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .set_table_styles([
                             # テーブル全体
                             {'selector': '', 'props': [('border-collapse', 'collapse'), ('font-family', 'Consolas, monospace'), ('font-size', '13px'), ('background-color', '#121212'), ('color', '#e0e0e0')]},
                             # ヘッダー行（Dark Gray / Green Accent）
                             {'selector': 'th', 'props': [('background-color', '#1f1f1f'), ('color', '#68a063'), ('border', '1px solid #333'), ('padding', '8px 12px'), ('text-align', 'center')]},
                             # データセル
                             {'selector': 'td', 'props': [('border', '1px solid #282828'), ('padding', '7px 12px'), ('text-align', 'right')]},
                             # ホバー効果
                             {'selector': 'tr:hover', 'props': [('background-color', '#1a261a')]}
                         ])
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [2]:
# =====================================================================
# CONFIGURATION & EXECUTION (Segmented by Market)
# =====================================================================
MARKET_TICKERS = {
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Singapore": {
        "STI Index": "^STI",
        "DBS Group": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
    },
    "Japan Stock": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG"
    }
}

# Run and display the segmented tables
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,"7,541.40",-0.51%,-2.62%,-1.76%,-2.99%,+3.89%,+10.25%
NASDAQ Composite,^IXIC,"25,164.36",-1.07%,-5.22%,-3.84%,-8.42%,-0.64%,+6.07%
Apple,AAPL,338.32,-1.83%,-1.60%,-1.63%,+1.63%,+19.26%,+25.59%
NVIDIA,NVDA,202.06,-1.26%,-6.18%,-6.45%,-15.42%,+3.06%,+0.03%
Microsoft,MSFT,507.12,-2.83%,-4.50%,-4.78%,-3.44%,+23.77%,-1.26%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"63,081.86",-0.33%,-1.36%,+1.43%,-7.65%,+4.20%,+22.95%
Toyota Motor,7203.T,"3,051.00",-0.62%,+0.00%,-2.12%,+6.81%,+1.17%,+0.10%
Sony Group,6758.T,"3,966.00",+0.20%,+0.23%,-4.09%,+5.68%,+25.00%,-7.29%
Mitsubishi UFJ Financial,8306.T,"3,383.00",-1.86%,+0.18%,+2.39%,-4.54%,+22.93%,+53.16%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,303.83",-1.36%,-1.74%,-2.07%,-6.36%,+7.64%,+19.34%
DBS Group,D05.SI,71.71,-1.94%,-3.08%,-3.37%,-7.86%,+24.88%,+40.49%
UOB,U11.SI,40.66,-1.72%,-2.12%,-1.55%,-5.57%,+13.33%,+21.52%
Singtel,Z74.SI,4.11,-0.48%,+0.74%,-0.96%,-3.75%,-8.40%,+0.39%
CapitaLand Ascendas REIT,A17U.SI,2.22,+0.45%,+1.83%,-0.45%,+0.00%,-10.21%,-17.23%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"6,213.00",+0.50%,-8.83%,-7.72%,-17.16%,-10.77%,+15.08%
ニデック (6594),6594.T,"1,762.00",-2.17%,-2.27%,+3.16%,-16.77%,-28.02%,-10.10%
未来工業 (7931),7931.T,"3,605.00",+0.14%,+5.41%,+3.44%,+1.12%,+23.71%,+12.41%
東部ネットワーク (9036),9036.T,"1,088.00",+0.37%,-2.07%,-1.98%,-10.23%,-9.86%,+14.11%
ニトリHD (9843),9843.T,"3,407.00",+0.12%,+3.15%,+0.12%,+13.23%,+51.14%,+37.11%
MRK HD (9980),9980.T,85.00,-2.30%,-2.30%,-2.30%,-4.49%,-15.00%,-3.58%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,96.16,-0.21%,+0.28%,+0.11%,+2.16%,-0.96%,-0.44%


In [3]:
MARKET_TICKERS = {
    "Singapore Healthcare": {
        # --- 病院経営・総合クリニック ---
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",  # 【修正】R01.SI から変更
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",  # 【修正】I01.SI から変更
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        
        # --- 医療不動産（REIT / 病院・介護施設） ---
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        
        # --- 専門医療・歯科クリニックチェーン ---
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",  # 【修正】1D8から変更
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        
        # --- 医療用消耗品・製薬 ---
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone": "AP4.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.81,-0.61%,-0.61%,+0.00%,-1.21%,-16.79%,-15.10%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.45,-1.21%,-1.21%,+0.00%,+0.41%,-11.18%,-1.06%
Thomson Medical Group (トムソン医療),A50.SI,0.05,+0.00%,+4.35%,+0.00%,-4.00%,-15.79%,-20.00%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,4.00,+0.50%,+0.76%,-1.48%,+1.27%,+2.37%,+2.27%
First REIT (ファースト・リート),AW9U.SI,0.17,-4.37%,-2.78%,-2.78%,-14.63%,-25.49%,-32.27%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.49,-1.02%,-1.02%,-2.02%,+4.30%,-15.04%,-7.94%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.18,+0.00%,+7.27%,+0.31%,+3.10%,+13.85%,+61.37%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.20,+0.00%,+0.00%,+0.00%,+1.01%,-9.67%,-8.52%
Medtecs International (医療用消耗品),546.SI,0.10,-1.96%,-5.66%,-7.41%,-12.28%,-14.53%,-20.00%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.36,+0.00%,+2.86%,+2.86%,-2.70%,+5.88%,+4.41%


In [4]:
MARKET_TICKERS = {
"Singapore Food & Agribusiness": {
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Riverstone": "AP4.SI",
        "ヤンジジャン・シップビルディング(BS6)": "BS6.SI",
        "Sembcorp Ind (U96)": "U96.SI",
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,2.95,-1.99%,-9.79%,-10.88%,-15.23%,-20.26%,-3.47%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,0.88,-0.57%,-2.23%,-2.78%,-5.91%,-23.91%,-4.79%
Riverstone,AP4.SI,0.73,-0.68%,-2.01%,+0.69%,-3.95%,+3.54%,-13.58%
ヤンジジャン・シップビルディング(BS6),BS6.SI,4.63,+1.76%,-3.34%,-4.14%,-8.86%,+11.84%,+37.92%
Sembcorp Ind (U96),U96.SI,5.82,-0.51%,-0.51%,-0.34%,-1.69%,-9.64%,-7.70%


<div style="background-color: #EBCB8B; color: #2E3440; padding: 12px; border-radius: 6px; font-weight: bold;">
</div>

In [5]:
MARKET_TICKERS = {
    "Singapore Stocks (Featured)": {
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Riverstone (リバーストーン・クリーンルーム用手袋)": "AP4.SI",
        "ヤンジジャン・シップビルディング (BS6)": "BS6.SI",
        "Sembcorp Ind (U96)": "U96.SI",
    }}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,2.95,-1.99%,-9.79%,-10.88%,-15.23%,-20.26%,-3.47%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,0.88,-0.57%,-2.23%,-2.78%,-5.91%,-23.91%,-4.79%
Riverstone (リバーストーン・クリーンルーム用手袋),AP4.SI,0.73,-0.68%,-2.01%,+0.69%,-3.95%,+3.54%,-13.58%
ヤンジジャン・シップビルディング (BS6),BS6.SI,4.63,+1.76%,-3.34%,-4.14%,-8.86%,+11.84%,+37.92%
Sembcorp Ind (U96),U96.SI,5.82,-0.51%,-0.51%,-0.34%,-1.69%,-9.64%,-7.70%


In [6]:
MARKET_TICKERS = {
    # 1. 安定重視・コア保有向け（好財務＋高配当）
    "Singapore Core & Stability (コア保有向け)": {
        "Sembcorp Industries (U96・大手公益/エネルギー)": "U96.SI",
        "Valuetronics Holdings (BN2・電子機器製造受託/無借金)": "BN2.SI",
    },
    # 2. 高利回り・インカムゲイン重視（配当狙い）
    "Singapore High Yield (高利回り狙い)": {
        "Multi-Chem (AWZ・ITセキュリティ/高配当)": "AWZ.SI",
        "Yangzijiang Financial Holding (YF8・金融)": "YF8.SI",
        "Yangzijiang Shipbuilding (BS6・造船/高収益)": "BS6.SI",
    },
    # 3. 高付加価値・成長＆分配金ハイブリッド
    "Singapore Growth & Hybrid (成長＆分配)": {
        "Riverstone Holdings (AP4・クリーンルーム用手袋)": "AP4.SI",
        "Parkway Life REIT (C2PU・医療REIT/安定分配)": "C2PU.SI",
        "Sanli Environmental Limited": "1E3.SI", 
    },
    
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Sembcorp Industries (U96・大手公益/エネルギー),U96.SI,5.82,-0.51%,-0.51%,-0.34%,-1.69%,-9.64%,-7.70%
Valuetronics Holdings (BN2・電子機器製造受託/無借金),BN2.SI,0.94,-0.53%,-2.08%,+0.00%,-6.00%,-13.56%,+15.06%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Multi-Chem (AWZ・ITセキュリティ/高配当),AWZ.SI,2.50,-5.66%,-1.96%,+7.30%,+2.88%,-29.41%,-20.15%
Yangzijiang Financial Holding (YF8・金融),YF8.SI,0.21,-2.33%,+6.60%,+8.25%,+9.95%,-19.23%,-80.56%
Yangzijiang Shipbuilding (BS6・造船/高収益),BS6.SI,4.63,+1.76%,-3.34%,-4.14%,-8.86%,+11.84%,+37.92%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Riverstone Holdings (AP4・クリーンルーム用手袋),AP4.SI,0.73,-0.68%,-2.01%,+0.69%,-3.95%,+3.54%,-13.58%
Parkway Life REIT (C2PU・医療REIT/安定分配),C2PU.SI,4.00,+0.50%,+0.76%,-1.48%,+1.27%,+2.37%,+2.27%
Sanli Environmental Limited,1E3.SI,0.08,-3.61%,-4.76%,-2.44%,-30.43%,-55.83%,-75.37%


In [7]:
MARKET_TICKERS = {
    # 1. 安定重視・コア保有向け（金融要塞・完全独占）
    "SG Core & Stability (コア保有向け)": {
        "DBS Group (D05.SI・金融要塞/東南アジア最大手/高ROE)": "D05.SI",
        "SGX Group (S68.SI・完全独占/国内唯一の証券取引所)": "S68.SI",
    },
    # 2. 高利回り・インカムゲイン重視（高配当・不動産仲介）
    "SG High Yield (高利回り狙い)": {
        "PropNex (OYY.SI・不動産仲介最大手/高配当・アセットライト)": "OYY.SI",
    },
    # 3. ディープバリュー・ネットキャッシュ
    "SG Value & Margin of Safety (バリュー・資産安全域)": {
        "Valuetronics (BN2.SI・高ネットキャッシュ/高い安全域)": "BN2.SI",
    },
    # 4. 業績復調・半導体関連（高配当＋資本利回り）
    "SG Recovery & Growth (復調・半導体工具)": {
        "Micro-Mechanics (508.SI・無借金/半導体消耗工具/高配当)": "508.SI",
    },
}

# 実行例
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
DBS Group (D05.SI・金融要塞/東南アジア最大手/高ROE),D05.SI,71.71,-1.94%,-3.08%,-3.37%,-7.86%,+24.88%,+40.49%
SGX Group (S68.SI・完全独占/国内唯一の証券取引所),S68.SI,20.32,+0.54%,-1.15%,-0.96%,-0.92%,-3.37%,+22.41%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
PropNex (OYY.SI・不動産仲介最大手/高配当・アセットライト),OYY.SI,1.65,+0.00%,-0.60%,-1.79%,-6.78%,-6.78%,-24.70%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Valuetronics (BN2.SI・高ネットキャッシュ/高い安全域),BN2.SI,0.94,-0.53%,-2.08%,+0.00%,-6.00%,-13.56%,+15.06%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Micro-Mechanics (508.SI・無借金/半導体消耗工具/高配当),508.SI,0.50,+0.00%,+0.00%,+0.00%,-18.70%,-11.50%,-14.50%


In [8]:
MARKET_TICKERS = {
    # 建設・インフラ関連銘柄（高配当・公共需要・改修）
    "SG Construction & Infrastructure (建設・インフラ系)": {
        "BRC Asia (1G6.SI・鉄筋加工最大手/高配当5-5.5%/PER7.5-8.5倍/公共住宅需要)": "BEC.SI",
        "OKP Holdings (5CF.SI・道路土木インフラ/PBR0.8倍割安/無借金経営/PER8-9.5倍)": "5CF.SI",
        "ISOTeam (5WF.SI・HDB建物修繕塗装R&R/小型ターンアラウンド/PER12-14倍)": "5WF.SI",
    },
}

# 実行例
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
BRC Asia (1G6.SI・鉄筋加工最大手/高配当5-5.5%/PER7.5-8.5倍/公共住宅需要),BEC.SI,4.13,-0.96%,+0.49%,+5.90%,+1.20%,-4.54%,+3.12%
OKP Holdings (5CF.SI・道路土木インフラ/PBR0.8倍割安/無借金経営/PER8-9.5倍),5CF.SI,0.62,+1.64%,+0.00%,+2.48%,-10.79%,-19.31%,+10.06%
ISOTeam (5WF.SI・HDB建物修繕塗装R&R/小型ターンアラウンド/PER12-14倍),5WF.SI,0.05,+0.00%,-3.64%,+0.00%,-17.19%,-34.57%,-34.73%


In [9]:
MARKET_TICKERS = {
   "USD ETF": {
        "Vanguard Funds PLC - Vanguard FTSE Global All-Cap UCITS ETF USD Acc (VALL.L)": "VALL.L",
},
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Vanguard Funds PLC - Vanguard FTSE Global All-Cap UCITS ETF USD Acc (VALL.L),VALL.L,nan,+nan%,+nan%,+nan%,+nan%,+nan%,+nan%


# Gemini Prompt
以下の【分析データ】を基に、指定の【Pythonコードフォーマット】に変換して出力してください。

【出力条件】
・MARKET_TICKERS の辞書形式で出力すること。
・辞書のキーには「企業名 (ティッカー・主要特徴/指標の要約)」を含めること。
・値（Value）には「ティッカーシンボル（.SI付き）」を入れること。
・解説文や導入の挨拶は不要で、そのままコピーして使えるPythonコードのみを出力すること。

【Pythonコードフォーマット例】
MARKET_TICKERS = {
    "カテゴリ名": {
        "銘柄名 (ティッカー・特徴要約)": "ティッカー.SI",
    },
}

【分析データ】
（※ここに分析したテキストや銘柄リスト、あるいは「以下の銘柄を分析してコード化して」と入力）